In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge

# Load train and test datasets
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f"Original Train shape: {train.shape}")
print(f"Original Test shape: {test.shape}")

Original Train shape: (1460, 81)
Original Test shape: (1459, 80)


In [5]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

train_raw = pd.read_csv('../data/train.csv')

def evaluate_cv(df_input):
    numeric_cols = df_input.select_dtypes(include=[np.number]).columns.drop(['Id', 'SalePrice'], errors='ignore')
    X = df_input[numeric_cols].fillna(0)
    y = np.log1p(df_input['SalePrice'])
    
    model = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
    scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=5)
    return -scores.mean()

score_with_outliers = evaluate_cv(train_raw)
print(f"CV Log RMSE WITH outliers:    {score_with_outliers:.5f}")

train_clean = train_raw[~train_raw['Id'].isin([524, 1299])].copy()

score_without_outliers = evaluate_cv(train_clean)
print(f"CV Log RMSE WITHOUT outliers: {score_without_outliers:.5f}")

print(f"Improvement (RMSE Drop):      {score_with_outliers - score_without_outliers:.5f}")

train = train_clean.reset_index(drop=True)

CV Log RMSE WITH outliers:    0.15368
CV Log RMSE WITHOUT outliers: 0.12921
Improvement (RMSE Drop):      0.02447


In [8]:
y_train = np.log1p(train['SalePrice'])
n_train = len(train)

# Drop Id and target to create a pure feature set for preprocessing
features = pd.concat([train.drop(['Id', 'SalePrice'], axis=1), 
                      test.drop(['Id'], axis=1)], axis=0).reset_index(drop=True)
print(f"Combined features shape: {features.shape}")

# 1. Missing means "Does Not Exist" (Fill with 'None')
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_cols:
    features[col] = features[col].fillna('None')

# 2. Genuine missing values (Fill with Mode / most common value)
mode_cols = ['Electrical', 'KitchenQual', 'Functional', 'SaleType', 'MSZoning', 'Exterior1st', 'Exterior2nd', 'Utilities']
for col in mode_cols:
    features[col] = features[col].fillna(features[col].mode()[0])

print(f"Remaining missing values in none_cols: {features[none_cols].isnull().sum().sum()}")
print(f"Remaining missing values in mode_cols: {features[mode_cols].isnull().sum().sum()}")

Combined features shape: (2917, 79)
Remaining missing values in none_cols: 0
Remaining missing values in mode_cols: 0


In [9]:
# 1. LotFrontage: Group by neighborhood and fill with median of that neighborhood
features['LotFrontage'] = features.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# 2. Garage-related numericals (Missing means no garage -> fill with 0)
garage_num_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars']
for col in garage_num_cols:
    features[col] = features[col].fillna(0)

# 3. Basement-related numericals (Missing means no basement -> fill with 0)
bsmt_num_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']
for col in bsmt_num_cols:
    features[col] = features[col].fillna(0)

# 4. Masonry veneer area (Missing means none -> fill with 0)
features['MasVnrArea'] = features['MasVnrArea'].fillna(0)

# Check if ANY missing values are left in the entire dataset
total_missing = features.isnull().sum().sum()
print(f"Total missing values remaining in ALL features: {total_missing}")

if total_missing > 0:
    # If there are still missing values, print out which columns have them
    missing_cols = features.isnull().sum()
    print(missing_cols[missing_cols > 0])
else:
    print("Awesome! The dataset is 100% clean of missing values!")

Total missing values remaining in ALL features: 0
Awesome! The dataset is 100% clean of missing values!


In [10]:
from scipy.stats import skew

# 1. Convert pseudo-numerical features to string (treat as categorical)
features['MSSubClass'] = features['MSSubClass'].apply(str)
features['YrSold'] = features['YrSold'].astype(str)
features['MoSold'] = features['MoSold'].astype(str)

# 2. Create a powerful new feature: Total Square Footage
features['TotalSF'] = features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']

# 3. Handle skewness in numerical features
# Identify all numerical features
numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numeric_features = features.select_dtypes(include=numeric_dtypes).columns

# Calculate skewness for all numerical features
skewed_feats = features[numeric_features].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewness = pd.DataFrame({'Skew': skewed_feats})

# Filter out highly skewed features (absolute skewness > 0.75)
skewness = skewness[abs(skewness['Skew']) > 0.75]
print(f"Number of highly skewed numerical features to log transform: {skewness.shape[0]}")

# Apply log1p transformation to these highly skewed features
skewed_features = skewness.index
for feat in skewed_features:
    features[feat] = np.log1p(features[feat])

# 4. One-Hot Encoding
# Convert all categorical text features into dummy variables (0/1)
final_features = pd.get_dummies(features).reset_index(drop=True)

print(f"Final feature matrix shape after one-hot encoding: {final_features.shape}")

Number of highly skewed numerical features to log transform: 21
Final feature matrix shape after one-hot encoding: (2917, 332)


In [11]:
import os

# 1. Split back into train and test sets using n_train index
X_train_clean = final_features.iloc[:n_train].copy()
X_test_clean = final_features.iloc[n_train:].copy()

print(f"Cleaned Train shape: {X_train_clean.shape}")
print(f"Cleaned Test shape: {X_test_clean.shape}")

# 2. Create target dataframe
y_train_df = pd.DataFrame({'SalePrice_log': y_train})

# 3. Create processed data directory if not exists
os.makedirs('../data/processed', exist_ok=True)

# 4. Save cleaned datasets to CSV files
X_train_clean.to_csv('../data/processed/train_clean.csv', index=False)
X_test_clean.to_csv('../data/processed/test_clean.csv', index=False)
y_train_df.to_csv('../data/processed/y_train.csv', index=False)

print("All processed datasets saved successfully to '../data/processed/'!")

Cleaned Train shape: (1458, 332)
Cleaned Test shape: (1459, 332)
All processed datasets saved successfully to '../data/processed/'!
